In [1]:
#undef __noinline__

# Thread Cooperation - The Grid-Stride Loop

In the previous notebooks each GPU thread handled exactly one array element.
That only works when you launch exactly as many threads as you have data points.

A more flexible pattern is the **grid-stride loop**: launch a fixed number of threads
(typically chosen to saturate the GPU) and have each thread walk through the array
in steps equal to the total number of threads in the grid.

This means:
- You never need to know N at launch time
- You can reuse the same launch config for any array size
- Each thread does a balanced share of the work

In [2]:
#include <cstdio>
#include <cmath>

static const int N   = 1 << 17;   // 131 072 elements
static const float ALPHA = 2.5f;     // scalar for SAXPY

__global__ void saxpy(float alpha, const float *x, float *y, int n) {
    int start  = threadIdx.x + blockIdx.x * blockDim.x;
    int step   = blockDim.x  * gridDim.x;          // total threads in the grid

    for (int i = start; i < n; i += step)
        y[i] = alpha * x[i] + y[i];
}

We launch with 64 blocks × 256 threads = 16 384 threads, but the array has 131 072 elements.
Each thread therefore handles roughly 8 elements via the loop. The result is verified
element-by-element on the CPU.

In [3]:
// Allocate, fill, launch, verifi
float *h_x = (float*)malloc(N * sizeof(float));
float *h_y = (float*)malloc(N * sizeof(float));
float *h_ref = (float*)malloc(N * sizeof(float));

for (int i = 0; i < N; i++) {
    h_x[i]   = sinf((float)i);
    h_y[i]   = cosf((float)i);
    h_ref[i] = ALPHA * h_x[i] + h_y[i];   // CPU reference
}

float *d_x, *d_y;
cudaMalloc(&d_x, N * sizeof(float));
cudaMalloc(&d_y, N * sizeof(float));
cudaMemcpy(d_x, h_x, N * sizeof(float), cudaMemcpyHostToDevice);
cudaMemcpy(d_y, h_y, N * sizeof(float), cudaMemcpyHostToDevice);

// 256 threads/block, 64 blocks — 16 384 total threads for 131 072 elements
saxpy<<<64, 256>>>(ALPHA, d_x, d_y, N);

cudaMemcpy(h_y, d_y, N * sizeof(float), cudaMemcpyDeviceToHost);

// Verify (allow small floating-point tolerance)
int errors = 0;
float max_err = 0.f;
for (int i = 0; i < N; i++) {
    float err = fabsf(h_y[i] - h_ref[i]);
    if (err > 1e-4f) errors++;
    if (err > max_err) max_err = err;
}

if (errors == 0)
    printf("Verified - %d elements correct, max error = %.2e\n", N, max_err);
else
    printf("FAIL - %d mismatches\n", errors);

cudaFree(d_x);  cudaFree(d_y);
free(h_x);  free(h_y);  free(h_ref);

Verified - 131072 elements correct, max error = 0.00e+00
